![Week 2 Day 4](../../assets/banners/w2d4.png)

# Week 2 Day 4 - SOLUTION - Citation Checking, RRF's k and Multi-Query

Reference solution for the exercise at the end of `week2/day4.ipynb`. Have a go yourself first: struggling a little is where the learning happens! Then compare.

**The exercise:** write `check_citations(answer, docs)`; explore RRF with `k=1` and `k=1000`; run multi-query on a lost-update question.

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"✅ OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("ℹ️ OpenAI API Key not set. That's OK for most of this lab: the offline cells use fakes.")
    print("   The few cells that call a real model will tell you so. Setup: Part 5 of the setup guide.")

In [ ]:
import re
import tempfile

from langchain_core.documents import Document
from langchain_core.embeddings import DeterministicFakeEmbedding

from gradewise.ingestion import get_vectorstore, ingest
from gradewise.rag import format_citation, get_retriever, reciprocal_rank_fusion, unique_citations

# ⚠️ Offline: fake embeddings in a temp folder, so we use BM25 (real) for retrieval
fake = DeterministicFakeEmbedding(size=64)
DB = tempfile.mkdtemp(prefix="gradewise_w2d4_solution_")
ingest(collection="cs201", embeddings=fake, persist_dir=DB, quiet=True)
store = get_vectorstore("cs201", embeddings=fake, persist_dir=DB)
bm25 = get_retriever(store=store, k=5, mode="bm25")

## Part 1: A cheap hallucination detector

In [ ]:
def check_citations(answer: str, docs: list[Document]) -> list[str]:
    """Citations in the answer that don't match any retrieved chunk (should be empty)."""
    allowed = set(unique_citations(docs))
    cited = re.findall(r"\[([^\[\]]+)\]", answer)
    return [c for c in cited if c.strip() not in allowed]


docs = bm25.invoke("What are the four necessary conditions for deadlock?")
good = f"Mutual exclusion, hold and wait, no pre-emption and circular wait [{format_citation(docs[0])}]."
bad = good + " Dijkstra proposed them in 1965 [unit1_operating_systems.pdf p9 §1.9 History]."

print("good answer →", check_citations(good, docs))
print("bad answer  →", check_citations(bad, docs))

The invented citation is caught: page 9 doesn't exist and was never retrieved. 🚀 **PRO:** in GradeWise you'd run this after every generated answer and flag (or regenerate) any answer with unsupported citations.

## Part 2: RRF's `k`

In [ ]:
def doc(name: str) -> Document:
    return Document(page_content=name, metadata={"source": "demo.pdf", "chunk_index": name})


# "solo" is 1st for retriever A only; "steady" is a modest 4th and 3rd, but found by BOTH
list_a = [doc("solo"), doc("x"), doc("y"), doc("steady")]
list_b = [doc("z"), doc("w"), doc("steady")]

for k in (1, 60, 1000):
    fused = reciprocal_rank_fusion([list_a, list_b], k=k)
    print(f"k = {k:<5} → {[d.page_content for d in fused[:3]]}")

With a **small k**, 1/(k + rank) falls off steeply, so single first places count for a lot (`solo` and `z` beat `steady` at k = 1). With a **large k**, every rank counts almost the same and agreement dominates. k = 60 is the classic middle ground.

## Part 3: Multi-query for a lost update

In [ ]:
question = "why is my database update lost when two people save at once?"
variants = [
    question,
    "lost update anomaly when two transactions write the same row concurrently",
    "isolation levels and locking prevent concurrency anomalies",
]
bm25_10 = get_retriever(store=store, k=10, mode="bm25")
print("Original question only:")
for d in bm25.invoke(question)[:4]:
    print("  ", format_citation(d))
print("Multi-query (3 phrasings, fused with RRF):")
for d in reciprocal_rank_fusion([bm25_10.invoke(v) for v in variants])[:4]:
    print("  ", format_citation(d))

The rewrites use the notes' own vocabulary (*lost update*, *transactions*, *isolation*), so the fused list lands on 3.6 Transactions and concurrency. With a key, generate the variants with the model:

In [ ]:
from pydantic import BaseModel, Field

from gradewise.errors import MissingAPIKeyError
from gradewise.models import chat_model


class QueryVariants(BaseModel):
    queries: list[str] = Field(description="3 different phrasings, using course terminology")


try:
    generated = (
        chat_model(temperature=0)
        .with_structured_output(QueryVariants)
        .invoke(f"Rewrite this student question 3 ways for searching CS201 course notes: {question}")
    )
    print(generated.queries)
    for d in reciprocal_rank_fusion([bm25_10.invoke(q) for q in [question, *generated.queries]])[:4]:
        print("  ", format_citation(d))
except MissingAPIKeyError as err:
    err.show()

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">RRF lives in <code>gradewise/rag/retrievers.py</code>; citations in <code>gradewise/rag/citations.py</code>.</span>
        </td>
    </tr>
</table>